In [ ]:
import sys
import os
# Automatically inject local virtual environment site-packages if running on Windows
venv_path = os.path.join(os.getcwd(), '.venv', 'Lib', 'site-packages')
if os.path.exists(venv_path) and venv_path not in sys.path:
    sys.path.insert(0, venv_path)

import os
if os.name == 'nt':  # Check if windows
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

import multiprocessing as mp
mp.freeze_support()  # For Windows  
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork


In [ ]:
'''
Explanation for state_dim=52:
TrackErrorDot = 1 (Distance from center of track)
TrackErrorCross = 1 (Left or Right)
Velocity = 3
HeadingAlignmentDot = 1 (Dot product of front vector and track vector)
HeadingAlignmentCross = 1 (Left or Right)
Jumping = 1
Rotation = 4
Distance = 1
Total = 13
Frame Stacking (x4) = 52
'''

import os
if os.name == 'nt':  # windows thing
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')
import pystk2
import numpy as np
from collections import deque

TRACK_HALF_WIDTH      = 5.0
STALL_SPEED_THRESHOLD = 2.0

class ProcessState:
	def __init__(self, PathNodes, max_speed=30, map_size=100, track_length=2000):
		self.frame        = deque(maxlen=4)
		self.max_speed    = max_speed
		self.map_size     = map_size
		self.track_length = track_length
		self.PathNodes    = np.array(PathNodes)[:, 0, :].astype(np.float32)
		self.LookaheadIndex = 5

	def processObservation(self, obs):
		KartLocation      = np.array(obs['location'], dtype=np.float32)
		KartFrontLocation = np.array(obs['front'],    dtype=np.float32)

		nodes_xz = self.PathNodes[:, [0, 2]]
		kart_xz  = KartLocation[[0, 2]]
		diff_sq  = np.sum((nodes_xz - kart_xz) ** 2, axis=1)
		AnchorNodeIndex = int(np.argmin(diff_sq))
		raw_error = float(np.sqrt(diff_sq[AnchorNodeIndex]))
		TrackErrorDot = np.array([np.clip(raw_error / TRACK_HALF_WIDTH, 0.0, 1.0)], dtype=np.float32)

		AnchorNode = self.PathNodes[AnchorNodeIndex]
		TargetNode = self.PathNodes[(AnchorNodeIndex + self.LookaheadIndex) % len(self.PathNodes)]

		track_vec_xz  = np.array([TargetNode[0] - AnchorNode[0], TargetNode[2] - AnchorNode[2]], dtype=np.float32)
		front_vec_xz  = np.array([KartFrontLocation[0] - KartLocation[0], KartFrontLocation[2] - KartLocation[2]], dtype=np.float32)
		offset_vec_xz = kart_xz - AnchorNode[[0, 2]]

		tv_norm = np.linalg.norm(track_vec_xz)
		fv_norm = np.linalg.norm(front_vec_xz)
		if tv_norm > 1e-6 and fv_norm > 1e-6:
			track_vec_xz /= tv_norm
			front_vec_xz /= fv_norm
			heading_dot   = float(np.dot(track_vec_xz, front_vec_xz))
			heading_cross = float(track_vec_xz[0] * front_vec_xz[1] - track_vec_xz[1] * front_vec_xz[0])
			track_cross   = float(track_vec_xz[0] * offset_vec_xz[1] - track_vec_xz[1] * offset_vec_xz[0])
		else:
			heading_dot = heading_cross = track_cross = 0.0

		HeadingAlignmentDot   = np.array([heading_dot],   dtype=np.float32)
		HeadingAlignmentCross = np.array([heading_cross], dtype=np.float32)
		TrackErrorCross       = np.array([track_cross],   dtype=np.float32)

		vel      = np.clip(np.array(obs['velocity'],  dtype=np.float32) / self.max_speed, -1.0, 1.0)
		jump     = np.array([1.0 if obs['jumping'] else 0.0], dtype=np.float32)
		rotation = np.array(obs['rotation'], dtype=np.float32)
		dist     = np.array([obs.get('distance_down_track', 0.0)], dtype=np.float32) / self.track_length

		state = np.concatenate([TrackErrorDot, TrackErrorCross, vel, HeadingAlignmentDot, HeadingAlignmentCross, jump, rotation, dist])

		if len(self.frame) == 0:
			for _ in range(4):
				self.frame.append(state)
		else:
			self.frame.append(state)

		return (np.concatenate(self.frame), TrackErrorDot[0], HeadingAlignmentDot[0])

def SingleInstance(rank, pipe):
	# hd() = full HD window, visible on your desktop
	pystk2.init(pystk2.GraphicsConfig.hd())
	WorldState = pystk2.WorldState()
	config = pystk2.RaceConfig(track='lighthouse', num_kart=1, laps=1)
	config.players[0].controller = pystk2.PlayerConfig.Controller.PLAYER_CONTROL
	race = pystk2.Race(config)
	try:
		race.start()
		track = pystk2.Track()
		track.update()
		track_length   = track.length
		max_coordinate = np.max(np.abs(track.path_nodes))
		processor = ProcessState(max_speed=30, map_size=max_coordinate, track_length=track_length, PathNodes=track.path_nodes)
		RaceEnded = False
		reward    = 0.0
		WorldState.update()
		kart      = WorldState.karts[0]
		prev_dist = kart.distance_down_track
		obs = {
			'location': kart.location,
			'velocity': kart.velocity,
			'front': kart.front,
			'jumping': kart.jumping,
			'rotation': kart.rotation,
			'distance_down_track': prev_dist
		}
		np_obs, TrackError, HeadingAlignment = processor.processObservation(obs=obs)
		# Send initial observation to model
		pipe.send([np_obs, reward, RaceEnded])

		StuckFrameCounter = 0

		while True:
			# Receive action from model
			ActionMessage = pipe.recv()

			if ActionMessage == 'TERMINATE':
				return

			action              = pystk2.Action()
			action.steer        = ActionMessage[0]
			action.acceleration = ActionMessage[1]
			# Brake only fires if braking signal is strong AND acceleration is low
			action.brake        = True if (ActionMessage[2] > 0.5 and ActionMessage[1] < 0.3) else False

			# Step the environment
			RaceEnded = not race.step(action)
			WorldState.update()

			kart         = WorldState.karts[0]
			current_dist = kart.distance_down_track

			obs = {
				'location': kart.location,
				'velocity': kart.velocity,
				'front': kart.front,
				'jumping': kart.jumping,
				'rotation': kart.rotation,
				'distance_down_track': current_dist
			}
			np_obs, TrackError, HeadingAlignment = processor.processObservation(obs=obs)

			# Reward Calculation
			vel_x, vel_y, vel_z = obs['velocity']
			speed = (vel_x**2 + vel_y**2 + vel_z**2)**0.5

			delta_dist = current_dist - prev_dist
			if abs(delta_dist) > 20.0:
				delta_dist = 0.0  # New lap or teleport — ignore

			reward = delta_dist * 80.0
			if delta_dist < -0.5:
				reward -= 8.0
			speed_factor = np.clip(speed / 10.0, 0.0, 1.0)
			reward -= TrackError * 4.0 * (1.0 + speed_factor)
			reward += HeadingAlignment * 2.0
			if speed < STALL_SPEED_THRESHOLD:
				reward -= 3.0
			if current_dist < 0:
				reward -= 5.0

			if speed < 2.0:
				StuckFrameCounter += 1
			else:
				StuckFrameCounter = 0

			if StuckFrameCounter > 60:
				reward    = -200.0
				RaceEnded = True

			# Send observation back to model
			pipe.send([np_obs, float(reward), RaceEnded])
			prev_dist = current_dist
	finally:
		# Critical Cleanup
		race.stop()
		del race
		pystk2.clean()


In [ ]:
# Instantiate models with state_dim=52 (13 features x 4 frame stack)
actor_net  = ActorNetwork(state_dim=52)
critic_net = CriticNetwork(state_dim=52)

def load_weights(net, path):
    """
    Shape-safe partial loader. strict=False alone still crashes on dimension
    mismatches, so we manually filter to only load tensors whose shapes match.
    New/expanded layers keep their random initialisation.
    """
    try:
        ckpt        = torch.load(path, map_location='cpu')
        model_state = net.state_dict()
        compatible  = {
            k: v for k, v in ckpt.items()
            if k in model_state and model_state[k].shape == v.shape
        }
        skipped = [k for k in ckpt if k not in compatible]
        model_state.update(compatible)
        net.load_state_dict(model_state)
        print(f'  [OK]   {path}: {len(compatible)}/{len(ckpt)} keys loaded.')
        if skipped:
            print(f'  [SKIP] {len(skipped)} key(s) skipped (shape mismatch — random init used).')
    except FileNotFoundError:
        print(f'  [WARN] {path} not found — running with fully random weights.')

load_weights(actor_net,  'best_actor.pth')
load_weights(critic_net, 'best_critic.pth')

# Set to evaluation mode for inference
actor_net.eval()
critic_net.eval()
print('Model ready.')

# Initialize Optimizer (kept for API compatibility, not used during inference)
optimizer = optim.Adam([
	{'params': actor_net.parameters(),  'lr': 1e-4},
	{'params': critic_net.parameters(), 'lr': 1e-4}
])


In [ ]:
def main():
	try:
		EpochLimit = 1

		for episode in range(EpochLimit):
			buffer               = []
			total_episode_reward = 0.0
			ProcessList          = []
			ConList              = []
			for i in range(1):
				ParentCon, ChildCon = mp.Pipe()
				process = mp.Process(target=SingleInstance, args=(i, ChildCon))
				ProcessList.append(process)
				ConList.append(ParentCon)
				process.start()

			BatchStates = []
			BatchDones  = []

			for con in ConList:
				np_obs, reward, RaceDone = con.recv()
				BatchStates.append(np_obs)
				BatchDones.append(RaceDone)

			for step in range(1000):

				# Build state tensor from worker observations
				state_tensor = torch.FloatTensor(np.array(BatchStates))

				# Sanity check for NaN from engine
				if torch.isnan(state_tensor).any():
					print('NaN detected in engine observations! Terminating episode.')
					break

				with torch.no_grad():
					action_dist = actor_net(state_tensor)
					# Use the mean (deterministic) for inference — no noise
					sampled_action = action_dist.mean
					state_value    = critic_net(state_tensor)
					BatchLogProbs  = action_dist.log_prob(sampled_action).sum(dim=-1)

				MemoryActions = []

				for i, con in enumerate(ConList):
					steer_val = torch.clamp(sampled_action[i, 0], min=-1.0, max=1.0).item()
					accel_val = torch.clamp(sampled_action[i, 1], min=0.0,  max=1.0).item()
					BrakeVal  = torch.clamp(sampled_action[i, 2], min=0.0,  max=1.0).item()
					MemoryActions.append((steer_val, accel_val, BrakeVal))

					if BatchDones[i]:
						con.send((0.0, 0.0, 0.0))  # Dummy vals for finished worker
					else:
						con.send((steer_val, accel_val, BrakeVal))

				NextStates      = []
				PreviousRewards = []
				PreviousDones   = []

				for con in ConList:
					np_obs, reward, RaceDone = con.recv()
					NextStates.append(np_obs)
					PreviousRewards.append(reward)
					PreviousDones.append(RaceDone)

				total_episode_reward += sum(PreviousRewards)

				BatchStates = NextStates
				BatchDones  = PreviousDones

				if all(PreviousDones):
					break

			# End of episode
			print(f'Episode: {episode + 1}/{EpochLimit} | Total Reward: {total_episode_reward:.2f} | Buffer Size: {len(buffer)}')

			for con in ConList:
				con.send('TERMINATE')
			for process in ProcessList:
				process.join()

	finally:
		for con in ConList:
			con.send('TERMINATE')
		for process in ProcessList:
			process.join()


In [ ]:
if __name__ == '__main__':
	main()
